# Blok 11: Python - Pobieranie danych (API, Web scraping)

**Cel notebooka:** przećwiczyć pobieranie danych z REST API (JSON) oraz web scraping (HTML) i zakończyć mini-projektem „Collector”.

**Zasady pracy:**
- Pracuj iteracyjnie: mały krok → sprawdź wynik → kolejny krok.
- Zawsze używaj `timeout` w requestach.
- Nie rób masowych pętli requestów do prawdziwych serwisów. Na Wikipedię: max 1–2 pobrania total.
- Ustaw `User-Agent` w nagłówkach.


## Opis „datasetu” (źródła danych)
W tym bloku nie ma jednego pliku-datasetu — pozyskujemy dane z kilku źródeł.

### Źródła API (JSON)
1) **Official Joke API**
- URL: https://official-joke-api.appspot.com
- Przykładowe endpointy: `/random_joke`, `/jokes/programming/ten`
- Typowy rekord (biznesowo: element contentowy do aplikacji/marketingu):
  - `id` (int) — identyfikator żartu
  - `type` (str) — kategoria
  - `setup` (str) — wprowadzenie
  - `punchline` (str) — puenta

2) **Hipolabs Universities API**
- URL: http://universities.hipolabs.com
- Endpoint: `/search` (parametry np. `country`, `name`)
- Typowy rekord (biznesowo: baza instytucji edukacyjnych, wzbogacanie CRM):
  - `name` (str) — nazwa uczelni
  - `country` (str) — kraj
  - `web_pages` (list[str]) — lista stron www
  - `domains` (list[str]) — domeny
  - `state-province` (str | null) — region

3) **PokeAPI**
- URL: https://pokeapi.co
- Endpoint: `/api/v2/pokemon/{name|id}`
- Rekord (biznesowo: przykład złożonego JSON; analogia do zagnieżdżonych danych produktowych):
  - `id` (int)
  - `name` (str)
  - `height` (int)
  - `weight` (int)
  - `types` (list[dict]) → docelowo: `types` (list[str])
  - `abilities` (list[dict]) → docelowo: `abilities` (list[str])

4) **SampleAPIs Futurama Episodes**
- URL: https://api.sampleapis.com/futurama/episodes
- Endpoint: `/episodes`
- Struktura może się zmieniać — **najpierw obejrzyj klucze** i dopiero potem buduj DataFrame.

### Źródła HTML (web scraping)
1) **W3Schools demo page**
- URL: https://w3schools.com/python/demopage.htm
- Cel: proste ćwiczenia selektorów, tytuł, linki, element po `id` / `class`.

2) **WebScraper test e-commerce**
- URL: https://webscraper.io/test-sites/e-commerce/allinone
- Cel: ekstrakcja listy produktów (biznesowo: monitoring cen/konkurencji na danych testowych)
- Docelowa tabela produktów:
  - `title` (str) — nazwa produktu
  - `price` (float) — cena
  - `description` (str) — krótki opis
  - `product_url` (str) — link do szczegółu (może być względny)
  - (opcjonalnie) `rating` (int/float) — jeśli uda się wyciągnąć

3) **Wikipedia** (używamy tylko do 1–2 spokojnych requestów)
- https://en.wikipedia.org/wiki/The_Game_Awards_2024
- https://en.wikipedia.org/wiki/96th_Academy_Awards
- Cel: pokazać, że HTML bywa złożony; wyciągamy mały fragment (np. nagłówek + pierwszą tabelę).


## Setup
Upewnij się, że masz zainstalowane biblioteki:
- `requests`, `pandas`, `beautifulsoup4` (opcjonalnie `lxml`)
- bonus: `playwright`

W trakcie ćwiczeń zapisujemy pliki do folderu `data/`.


In [1]:
from __future__ import annotations

import os
import json
import time
from datetime import datetime, timezone
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

DEFAULT_HEADERS = {
    "User-Agent": "ds-ai-course/1.0 (+educational)"
}

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print("OK: imports + data directory ready")


OK: imports + data directory ready


## Pomocnicze funkcje (szkielety)
W pracy produkcyjnej zwykle rozdzielamy: fetch (pobranie) → parse (parsowanie) → validate → save.

Twoje zadanie: uzupełnić TODO, ale nie „przekombinować” — prosto i czytelnie.


In [2]:
def fetch_json(url: str, 
               params: dict| None =None, 
               headers: dict | None =None, 
               timeout: int=10) -> dict | list:
    """Pobierz JSON z URL i zwróć jako dict/list.
    
    - połącz nagłówki DEFAULT_HEADERS z headers przekazanym w argumencie
    - wykonaj request GET z timeout
    - wypisz status_code i reason
    - jeśli błąd (4xx/5xx), rzuć wyjątek (np. raise_for_status)
    - zwróć resp.json()
    """
    headers_all = DEFAULT_HEADERS
    if headers:
        headers_all.update(headers)

    resp = requests.get(url, params=params, headers=headers_all, timeout=timeout)
    print(f'Status: {resp.status_code}')
    print(f'Reason: {resp.reason}')

    resp.raise_for_status()

    return resp.json()


def fetch_html(url: str, 
               headers: dict | None =None, 
               timeout: int =10) -> str:
    """Pobierz HTML jako tekst.
    
    - ustaw User-Agent
    - dodaj timeout
    - obsłuż błędy statusu
    - zwróć resp.text
    """
    headers_all = DEFAULT_HEADERS
    if headers:
        headers_all.update(headers)
    
    resp = requests.get(url, headers=headers_all, timeout=timeout)
    
    resp.raise_for_status()

    return resp.text

def utc_now_iso() -> str:
    """Timestamp UTC w ISO8601."""
    return datetime.now(timezone.utc).isoformat()


def safe_text(el) -> str:
    """Zwraca tekst elementu HTML albo pusty string, jeśli element nie istnieje."""
    return "" if el is None else el.get_text(strip=True)


def safe_attr(el, attr: str) -> str:
    """Zwraca atrybut elementu HTML albo pusty string, jeśli nie istnieje."""
    return "" if el is None else el.get(attr)


In [ ]:
fetch_json(url="https://official-joke-api.appspot.com/jokes/random")

Status: 200
Reason: OK


{'type': 'programming',
 'setup': 'Why do Java programmers wear glasses?',
 'punchline': "Because they don't C#.",
 'id': 34}

# Ćwiczenia
Łącznie: **15 zadań**. W większości to praktyka. Przy każdym zadaniu: najpierw zrób minimalny działający wariant, potem dopiero „upiększaj”.

Sugestia: po każdym zadaniu zapisuj wynik do zmiennej i pokaż `head()` / `len()` / przykładowe rekordy.


## Z1 (łatwe, ~10 min) — Hello API (Official Joke)
**Biz:** pobierz 1 żart (jako przykład „rekordu contentowego”).

**Cel techniczny:** `requests.get`, status code, `json()`.

TODO:
- Pobierz `random_joke`
- Wypisz `status_code` i `reason`
- Wypisz `setup` i `punchline`


In [8]:
JOKE_URL = "https://official-joke-api.appspot.com/random_joke"

data = fetch_json(JOKE_URL)
print(data, "\n")
print(data["setup"], "\n", data["punchline"])

Status: 200
Reason: OK
{'type': 'general', 'setup': 'Want to hear a joke about a piece of paper?', 'punchline': "Never mind...it's tearable", 'id': 39} 

Want to hear a joke about a piece of paper? 
 Never mind...it's tearable


## Z2 (łatwe, ~12 min) — Parametry w zapytaniu (Universities)
**Biz:** lista uczelni dla kraju + opcjonalny filtr po fragmencie nazwy.

**Cel techniczny:** `params={...}` i praca na liście rekordów.

TODO:
- Pobierz uczelnie dla `country="Poland"`
- Sprawdź ile rekordów przyszło
- Wypisz 3 przykładowe nazwy uczelni


In [28]:
UNI_URL = "http://universities.hipolabs.com/search"
params = {
    "country": "Poland",
    "name": "Gdansk"
}

data = fetch_json(UNI_URL, params=params)
print(data)
print(type(data), len(data))
print([row["name"] for row in data[:3]])

Status: 200
Reason: OK
[{'country': 'Poland', 'alpha_two_code': 'PL', 'state-province': None, 'web_pages': ['http://www.wsz.pl/'], 'name': 'Gdansk Management College', 'domains': ['wsz.pl']}, {'country': 'Poland', 'alpha_two_code': 'PL', 'state-province': None, 'web_pages': ['http://www.univ.gda.pl/'], 'name': 'University of Gdansk', 'domains': ['univ.gda.pl']}, {'country': 'Poland', 'alpha_two_code': 'PL', 'state-province': None, 'web_pages': ['http://www.pg.gda.pl/'], 'name': 'Technical University of Gdansk', 'domains': ['pg.gda.pl']}, {'country': 'Poland', 'alpha_two_code': 'PL', 'state-province': None, 'web_pages': ['http://www.awf.gda.pl/'], 'name': 'Physical Education Academy "Jedrzej Sniadecki" in Gdansk', 'domains': ['awf.gda.pl']}, {'country': 'Poland', 'alpha_two_code': 'PL', 'state-province': None, 'web_pages': ['http://www.asp.gda.pl/'], 'name': 'Fine Arts Academy in Gdansk', 'domains': ['asp.gda.pl']}, {'country': 'Poland', 'alpha_two_code': 'PL', 'state-province': None, '

## Z3 (łatwe/średnie, ~12 min) — Universities → DataFrame
**Biz:** zamieniamy API na tabelę, żeby móc filtrować/sortować.

**Cel techniczny:** `pd.DataFrame(...)`, wybór kolumn.

TODO:
- Stwórz DataFrame z wyniku Z2
- Wybierz kolumny: `name`, `country`, `web_pages`, `domains`, `state-province`
- Pokaż `head(10)`


In [29]:
df_uni = pd.DataFrame(data)
cols = ["name", "country", "web_pages", "domains", "state-province"]
df_uni[cols].head(10)

,name,country,web_pages,domains,state-province
0,Gdansk Management College,Poland,[http://www.wsz.pl/],[wsz.pl],None
1,University of Gdansk,Poland,[http://www.univ.gda.pl/],[univ.gda.pl],None
2,Technical University of Gdansk,Poland,[http://www.pg.gda.pl/],[pg.gda.pl],None
3,"Physical Education Academy ""Jedrzej Sniadecki""...",Poland,[http://www.awf.gda.pl/],[awf.gda.pl],None
4,Fine Arts Academy in Gdansk,Poland,[http://www.asp.gda.pl/],[asp.gda.pl],None
5,Medical Academy in Gdansk,Poland,[http://www.amg.gda.pl/],[amg.gda.pl],None
6,"Music Academy ""Stanislaw Moniuszko"" in Gdansk",Poland,[http://www.amuz.gda.pl/],[amuz.gda.pl],None


## Z4 (średnie, ~15 min) — Zagnieżdżony JSON (PokeAPI)
**Biz:** analogia do zagnieżdżonych danych produktowych (warianty, atrybuty).

**Cel techniczny:** wybieranie pól z nested JSON + list comprehensions.

TODO:
- Pobierz pokemona po nazwie (np. `ditto`)
- Zbuduj prosty dict z polami: `id`, `name`, `height`, `weight`, `types` (lista stringów)
- (opcjonalnie) dodaj `abilities` (lista stringów)


In [40]:
POKE_URL = "https://pokeapi.co/api/v2/pokemon/ditto"

data = fetch_json(POKE_URL)
pokemon = {
    "id": data["id"],
    "name": data["name"],
    "height": data["height"],
    "weight": data["weight"],
    "types": [x["type"]["name"] for x in data["types"]],
    "abilities": [x["ability"]["name"] for x in data["abilities"]]
}
pokemon

Status: 200
Reason: OK


{'id': 132,
 'name': 'ditto',
 'height': 3,
 'weight': 40,
 'types': ['normal'],
 'abilities': ['limber', 'imposter']}

## Z5 (średnie, ~15 min) — Obsługa błędów: 404 z czytelnym komunikatem
**Biz:** user wpisuje nazwę produktu/obiektu; jeśli nie istnieje, dajesz sensowną informację.

**Cel techniczny:** `try/except` + `raise_for_status()`.

TODO:
- Napisz funkcję `get_pokemon_basic(name: str) -> dict | None`
- Jeśli dostaniesz 404, zwróć `None` i wypisz komunikat „Pokemon not found”
- Jeśli inne błędy: rzuć wyjątek dalej


In [ ]:
def get_pokemon_basic(name: str):
    # TODO: zaimplementuj
    # hint: url = f"https://pokeapi.co/api/v2/pokemon/{name}"
    return None


# TODO: przetestuj na poprawnej i błędnej nazwie
# print(get_pokemon_basic("ditto"))
# print(get_pokemon_basic("THIS_DOES_NOT_EXIST"))


## Z6 (łatwe/średnie, ~12 min) — Batch pobierania (mała skala)
**Biz:** generujesz małą paczkę danych do prototypu (np. 10 elementów do testu modelu/analizy).

**Cel techniczny:** pętla + lista rekordów.

TODO:
- Pobierz 10 żartów (np. endpoint `.../jokes/programming/ten` albo 10× `random_joke`)
- Zapisz jako listę dictów
- Pokaż ile rekordów zebrano

Uwaga: jeśli robisz 10× `random_joke`, dodaj małe opóźnienie `time.sleep(0.2)`.


In [ ]:
# TODO
# OPTION A: endpoint /jokes/programming/ten
# url = "https://official-joke-api.appspot.com/jokes/programming/ten"
# jokes = fetch_json(url)
# print(type(jokes), len(jokes))

# OPTION B: 10x random_joke
# jokes = []
# for _ in range(10):
#     jokes.append(fetch_json(JOKE_URL))
#     time.sleep(0.2)
# print(len(jokes))

## Z7 (średnie, ~15 min) — Zapis danych (CSV i JSON)
**Biz:** pipeline: pobierasz → zapisujesz „raw/processed” na dysk.

**Cel techniczny:** `to_csv`, `to_json` + metadane pobrania.

TODO:
- Zrób DataFrame z żartów z Z6
- Dodaj kolumny: `source` i `fetched_at` (UTC)
- Zapisz do `data/jokes.csv` i `data/jokes.json`


In [ ]:
# TODO
# df_jokes = pd.DataFrame(jokes)
# df_jokes["source"] = "official-joke-api"
# df_jokes["fetched_at"] = utc_now_iso()

# csv_path = os.path.join(DATA_DIR, "jokes.csv")
# json_path = os.path.join(DATA_DIR, "jokes.json")

# df_jokes.to_csv(csv_path, index=False)
# df_jokes.to_json(json_path, orient="records", force_ascii=False)
# print("Saved:", csv_path, json_path)

## Z8 (średnie, ~18 min) — Futurama episodes: szybka analiza
**Biz:** prosta metryka: ile odcinków na sezon / średnia ocena / cokolwiek, co ma sens w danych.

**Cel techniczny:** najpierw eksploracja kluczy, potem DataFrame i `groupby`.

TODO:
- Pobierz `https://api.sampleapis.com/futurama/episodes`
- Obejrzyj 1 rekord: jakie ma klucze?
- Stwórz DataFrame
- Znajdź kolumnę sezonu (albo numeru) i policz: ile odcinków w grupie

Hint: jeśli nie ma sezonu, zrób inną sensowną agregację (np. po roku w dacie emisji, jeśli jest).


In [ ]:
# TODO
FUTURAMA_URL = "https://api.sampleapis.com/futurama/episodes"

# eps = fetch_json(FUTURAMA_URL)
# print(type(eps), len(eps))
# print("Example keys:", list(eps[0].keys()) if isinstance(eps, list) and eps else None)

# df_eps = pd.DataFrame(eps)
# df_eps.head()

## Z9 (łatwe, ~10 min) — Pierwszy scraping: tytuł strony i linki (W3Schools demo)
**Biz:** automatyczne zbieranie linków/treści do monitoringu/indeksu.

**Cel techniczny:** BeautifulSoup, `.find`, `.find_all`, atrybut `href`.

TODO:
- Pobierz HTML z `https://w3schools.com/python/demopage.htm`
- Wyciągnij tytuł strony (spróbuj `title` i/lub `h1`)
- Zbierz wszystkie linki (`<a>`) jako lista URL
- Pokaż 10 pierwszych linków


In [53]:
# TODO
DEMO_URL = "https://w3schools.com/python/demopage.htm"

html = fetch_html(DEMO_URL)
soup = BeautifulSoup(html, "html.parser")
page_title = soup.select('title')
links = []
print(page_title)
# print(links[:10])

[]


## Z10 (łatwe, ~12 min) — Wyszukiwanie po `id` i `class`
**Biz:** szybkie odnajdywanie kluczowych elementów (np. panel ceny, komunikat, status dostępności).

**Cel techniczny:** `.find(id=...)`, `.find(class_=...)`.

TODO:
- Na demopage znajdź 1 element po `id`
- Znajdź 1 element po `class`
- Wypisz tekst elementów

Hint: najpierw obejrzyj HTML w przeglądarce (Inspect) i wybierz sensowny `id`/`class`.


In [58]:
RANDOM_WIKI_URL = "https://pl.wikipedia.org/wiki/Specjalna:Losowa_strona"

html = fetch_html(RANDOM_WIKI_URL)
soup = BeautifulSoup(html, "html.parser")

el_by_id = soup.find(id="1")
el_by_class = soup.find(class_="2")
print(safe_text(el_by_id))
print(safe_text(el_by_class))

## Z11 (średnie, ~18 min) — WebScraper test e-commerce: lista produktów
**Biz:** monitoring cen i opisów na danych testowych.

**Cel techniczny:** `select/select_one`, iteracja po „kartach” produktu.

TODO:
- Pobierz HTML: `https://webscraper.io/test-sites/e-commerce/allinone`
- Znajdź listę kart produktu (spróbuj selektora CSS na kontener produktu)
- Dla każdej karty wyciągnij: `title`, `price` (string), `description`, `product_url` (może być względny)
- Zbuduj listę dictów `products`
- Pokaż: liczba produktów i 3 pierwsze rekordy


In [3]:
# TODO
SHOP_URL = "https://webscraper.io/test-sites/e-commerce/allinone"

html = fetch_html(SHOP_URL)
soup_shop = BeautifulSoup(html, "html.parser")

cards = soup_shop.select("div")
# products = []
# for card in cards:
#     title = ""  # TODO
#     price_raw = ""  # TODO
#     desc = ""  # TODO
#     href = ""  # TODO
#     product_url = urljoin(SHOP_URL, href) if href else ""
#     products.append({
#         "title": title,
#         "price_raw": price_raw,
#         "description": desc,
#         "product_url": product_url,
#     })

# print("cards:", len(cards), "products:", len(products))
# products[:3]

In [4]:
cards

[<div class="v2-announcement alert alert-dismissible fade show m-0" role="alert">
 <div class="container text-center">
 <a class="v2-announcement__link" href="https://cloud.webscraper.io/register?utm_source=cloud-landing&amp;utm_medium=announcement-bar">
 <span>Web Scraper Cloud</span>
 <span class="full-announcement d-none d-lg-inline">replaces your complex scraping stack.</span>
 <span class="mobile-announcement d-inline d-lg-none">-</span>
 <span class="v2-announcement-try">Try it free for 7 days</span>
 <svg class="v2-button-icon-right" fill="none" height="18" viewbox="0 0 18 18" width="18" xmlns="http://www.w3.org/2000/svg">
 <path d="M10 14.25L15 9L10 3.75M15 9L3 9" stroke="#00303D" stroke-linecap="round" stroke-linejoin="round" stroke-width="1.4"></path>
 </svg>
 </a>
 <button aria-label="Close" class="v2-announcement__close d-none d-lg-inline" data-bs-dismiss="alert" type="button">
 <svg fill="none" height="18" viewbox="0 0 18 18" width="18" xmlns="http://www.w3.org/2000/svg">


## Z12 (średnie, ~15 min) — Czyszczenie ceny i metryki
**Biz:** bez czyszczenia danych nie zrobisz sensownej analizy (np. średnia cena, percentyle).

**Cel techniczny:** czyszczenie stringów → float + prosta statystyka.

TODO:
- Zrób DataFrame z `products`
- Zamień `price_raw` na `price` (float)
- Policz średnią cenę i medianę
- Pokaż 5 najdroższych produktów


In [ ]:
# TODO
# df_prod = pd.DataFrame(products)
# df_prod.head()

# TODO: wyczyść df_prod["price_raw"] → df_prod["price"] jako float
# hint: usuń znaki waluty i spacje

# print("mean:", df_prod["price"].mean())
# print("median:", df_prod["price"].median())
# df_prod.sort_values("price", ascending=False).head(5)

## Z13 (średnie, ~15 min) — Ekstrakcja URL-i i walidacja
**Biz:** link do produktu to często klucz do dalszego pobierania (szczegóły, dostępność, warianty).

**Cel techniczny:** sprawdzenie jakości danych + deduplikacja.

TODO:
- Sprawdź ile `product_url` jest pustych
- Usuń duplikaty po `product_url` (jeśli są)
- Zapisz produkty do pliku `data/products.csv`


In [ ]:
# TODO
# empty_urls = (df_prod["product_url"].fillna("") == "").sum()
# print("Empty URLs:", empty_urls)

# df_prod = df_prod.drop_duplicates(subset=["product_url"]).reset_index(drop=True)

# out_path = os.path.join(DATA_DIR, "products.csv")
# df_prod.to_csv(out_path, index=False)
# print("Saved:", out_path)

## Z14 (BONUS, ~18 min) — Playwright: render strony i parsuj HTML
**Kiedy to potrzebne:** gdy `requests.get(url).text` nie ma danych (bo generuje je JavaScript).

**Cel techniczny:** minimalny przepływ: Playwright → `page.content()` → BeautifulSoup.

TODO:
- Zainstaluj: `pip install playwright` i uruchom w terminalu: `playwright install`
- Użyj Playwright, żeby pobrać HTML po renderze
- Sparsuj BS4 i wyciągnij np. liczbę kart produktu

Jeśli nie masz Playwright: pomiń zadanie.


In [ ]:
# TODO (BONUS)
# Jeśli korzystasz z Jupytera, pamiętaj o problemach z asyncio.
# Najprościej: uruchom to jako skrypt w .py albo użyj nest_asyncio.

# from playwright.async_api import async_playwright

# async def fetch_rendered_html(url: str) -> str:
#     async with async_playwright() as p:
#         browser = await p.chromium.launch(headless=True)
#         page = await browser.new_page()
#         await page.goto(url)
#         await page.wait_for_timeout(1000)
#         html = await page.content()
#         await browser.close()
#         return html

# html = await fetch_rendered_html(SHOP_URL)
# soup = BeautifulSoup(html, "html.parser")
# print(len(soup.select("div.thumbnail")))

## Z15 (mini-projekt, ~25 min) — Collector: 2 źródła → zapis + metadane
**Biz:** budujesz mały kolektor danych, który potrafi pobrać dane z 2 różnych źródeł i zapisać je spójnie do folderu `data/`.

**Wymagania:**
1) Źródło A (API): wybierz jedno
- Universities (Poland)
- Futurama episodes
- Jokes

2) Źródło B (HTML scraping): produkty z test e-commerce

**Co zapisujesz:**
- `data/collector_api.csv` i/lub `data/collector_api.json`
- `data/collector_products.csv` i/lub `data/collector_products.json`
- Dla obu: dodaj kolumny `source`, `fetched_at`

**Walidacja (minimum):**
- jeśli DataFrame jest pusty → przerwij z czytelnym komunikatem
- sprawdź, czy istnieją wymagane kolumny (np. dla produktów: `title`, `price`)

TODO:
- Zaimplementuj funkcje `collect_api()` i `collect_products()`
- Uruchom i zapisz wyniki


In [ ]:
def collect_api() -> pd.DataFrame:
    """TODO: pobierz dane z wybranego API i zwróć jako DataFrame z metadanymi."""
    # TODO: wybierz źródło i zaimplementuj
    # hint: df["source"] = "..."; df["fetched_at"] = utc_now_iso()
    raise NotImplementedError


def collect_products() -> pd.DataFrame:
    """TODO: pobierz produkty z test e-commerce i zwróć jako DataFrame z metadanymi."""
    # TODO: użyj fragmentów z Z11–Z13
    raise NotImplementedError


def validate_non_empty(df: pd.DataFrame, required_cols: list[str] | None = None) -> None:
    """Walidacja minimum: niepusty df + wymagane kolumny."""
    # TODO
    return None


# TODO: uruchom Collector
# df_api = collect_api()
# validate_non_empty(df_api)

# df_products = collect_products()
# validate_non_empty(df_products, required_cols=["title", "price"])

# TODO: zapisz pliki
# df_api.to_csv(os.path.join(DATA_DIR, "collector_api.csv"), index=False)
# df_products.to_csv(os.path.join(DATA_DIR, "collector_products.csv"), index=False)
# print("Collector done!")

## (Opcjonalnie) Wikipedia — micro-scraping
Jeśli macie czas i ochotę, zróbcie **jedno** pobranie jednej strony i wyciągnijcie:
- tytuł (`h1`)
- liczbę tabel (`table`)
- pierwsze 5 wierszy z pierwszej sensownej tabeli (jeśli to ma sens)

Uwaga: Wikipedia może blokować agresywne scrapowanie. Tu robimy tylko minimalny przykład.


In [ ]:
# TODO (opcjonalnie)
# WIKI_URL = "https://en.wikipedia.org/wiki/The_Game_Awards_2024"
# html = fetch_html(WIKI_URL)
# soup = BeautifulSoup(html, "html.parser")
# print("H1:", safe_text(soup.select_one("h1")))
# tables = soup.select("table")
# print("Tables:", len(tables))
# # TODO: spróbuj sparsować 1 tabelę do DataFrame (np. pd.read_html) — ale ostrożnie z wagą strony

# Podsumowanie
Jeśli dotarłeś(aś) tutaj, to masz już najważniejsze nawyki:
- request z `timeout` + `User-Agent`
- kontrola `status_code` i obsługa błędów
- JSON → dict/list → DataFrame
- HTML → BeautifulSoup → selektory → lista rekordów → DataFrame
- zapis danych + metadane (`source`, `fetched_at`)

Następny krok w realnych projektach: dołożyć cache, retry/backoff i testy jakości danych.
